# Naive Bayes: Play Tennis Example Without Encoders

This notebook uses the classic **Play Tennis** dataset.

Important: This version does **not** use LabelEncoder, OneHotEncoder, or OrdinalEncoder.

We calculate probabilities manually using pandas.

In [ ]:
import pandas as pd


## Step 1: Create the Dataset

In [ ]:
data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}

df = pd.DataFrame(data)
df

## Step 2: Check Class Counts

We first count how many times `Yes` and `No` appear in the target column.

In [ ]:
df['PlayTennis'].value_counts()

## Step 3: Calculate Prior Probabilities

Prior probability means probability of each class before looking at features.

$$P(Yes) = \frac{Total\ Yes}{Total\ Records}$$

$$P(No) = \frac{Total\ No}{Total\ Records}$$

In [ ]:
total = len(df)
yes_count = len(df[df['PlayTennis'] == 'Yes'])
no_count = len(df[df['PlayTennis'] == 'No'])

p_yes = yes_count / total
p_no = no_count / total

print('Total Records:', total)
print('Yes Count:', yes_count)
print('No Count:', no_count)
print('P(Yes):', p_yes)
print('P(No):', p_no)

## Step 4: Conditional Probability Function

This function calculates:

$$P(Feature\ Value | Class)$$

Example:

$$P(Sunny | Yes)$$

This means: among all rows where PlayTennis is Yes, how many have Outlook = Sunny?

In [ ]:
def conditional_probability(feature, value, target_class):
    class_rows = df[df['PlayTennis'] == target_class]
    count_value_in_class = len(class_rows[class_rows[feature] == value])
    total_class_count = len(class_rows)
    return count_value_in_class / total_class_count

print('P(Sunny | Yes):', conditional_probability('Outlook', 'Sunny', 'Yes'))
print('P(Sunny | No):', conditional_probability('Outlook', 'Sunny', 'No'))

## Step 5: Test One New Example

We will predict:

**Outlook = Sunny, Temperature = Hot, Humidity = High, Wind = Weak**

Now calculate:

$$P(Yes | Sunny, Hot, High, Weak)$$

and

$$P(No | Sunny, Hot, High, Weak)$$

In [ ]:
new_data = {
    'Outlook': 'Sunny',
    'Temperature': 'Hot',
    'Humidity': 'High',
    'Wind': 'Weak'
}

new_data

## Step 6: Calculate Probability for Yes and No

Naive Bayes formula:

$$P(Yes|X) \propto P(Yes) \times P(Outlook|Yes) \times P(Temperature|Yes) \times P(Humidity|Yes) \times P(Wind|Yes)$$

$$P(No|X) \propto P(No) \times P(Outlook|No) \times P(Temperature|No) \times P(Humidity|No) \times P(Wind|No)$$

In [ ]:
p_yes_given_x = p_yes
p_no_given_x = p_no

for feature, value in new_data.items():
    p_yes_given_x *= conditional_probability(feature, value, 'Yes')
    p_no_given_x *= conditional_probability(feature, value, 'No')

print('Unnormalized P(Yes | X):', p_yes_given_x)
print('Unnormalized P(No | X):', p_no_given_x)

## Step 7: Normalize the Probabilities

To convert them into understandable percentages:

$$P(Yes|X) = \frac{Score\ Yes}{Score\ Yes + Score\ No}$$

$$P(No|X) = \frac{Score\ No}{Score\ Yes + Score\ No}$$

In [ ]:
total_score = p_yes_given_x + p_no_given_x

final_yes_probability = p_yes_given_x / total_score
final_no_probability = p_no_given_x / total_score

print('Final P(Yes | X):', final_yes_probability)
print('Final P(No | X):', final_no_probability)

if final_yes_probability > final_no_probability:
    print('Prediction: Play Tennis = Yes')
else:
    print('Prediction: Play Tennis = No')

## Step 8: Create a Reusable Prediction Function

Now we will make a function so we can test any new input easily.

In [ ]:
def predict_play_tennis(outlook, temperature, humidity, wind):
    sample = {
        'Outlook': outlook,
        'Temperature': temperature,
        'Humidity': humidity,
        'Wind': wind
    }
    
    yes_score = p_yes
    no_score = p_no
    
    for feature, value in sample.items():
        yes_score *= conditional_probability(feature, value, 'Yes')
        no_score *= conditional_probability(feature, value, 'No')
    
    total_score = yes_score + no_score
    yes_probability = yes_score / total_score
    no_probability = no_score / total_score
    
    prediction = 'Yes' if yes_probability > no_probability else 'No'
    
    return {
        'P(Yes)': yes_probability,
        'P(No)': no_probability,
        'Prediction': prediction
    }

## Step 9: Test More Examples

In [ ]:
predict_play_tennis('Sunny', 'Hot', 'High', 'Weak')

In [ ]:
predict_play_tennis('Overcast', 'Mild', 'High', 'Strong')

In [ ]:
predict_play_tennis('Rain', 'Mild', 'High', 'Strong')

## Student Practice Task

Predict the result for:

**Outlook = Overcast, Temperature = Mild, Humidity = Normal, Wind = Weak**

Then compare your answer with the function output.